In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd

from ai_kucharka import (
    apply_conversion,
    convert_materials,
    expand_rows,
    export_dataframe,
    find_images,
    load_eda_composition,
    load_material_conversion,
    process_files,
    remove_nonunique,
    remove_strings,
)

In [ ]:
import os
import tomllib

with open("../configs/company123.toml", "rb") as f:
    config = tomllib.load(f)

root = os.path.join("..", config["root_output"])
root_input = os.path.join("..", config["root_input"])
replace_source = [tuple(x) for x in config.get("replace_source", [])]

In [ ]:
conversion = load_material_conversion()
conversion_lower = {x["COMTES MATERIAL VÝBĚR"].lower(): x["EDA MMM"] for _, x in conversion.iterrows()}
composition = load_eda_composition()

ignore_str = {"x", "eda", "", ":", "-"}
cols_expand = ["Test", "Material", "Material EDA", "Composition"]
cols_order = [
    "File",
    "Test",
    "Material",
    "Material EDA",
    "Images",
    "Composition",
    "Composition EDA",
]

results, tests = process_files(root)

In [ ]:
results["Material"] = results["Material"].apply(lambda x: remove_strings(x, ignore_str=ignore_str))
results["Test"] = results["Test"].apply(lambda x: remove_strings(x, ignore_str=ignore_str))
results["Composition"] = [convert_materials(row["Composition"]) for _, row in results.iterrows()]
results["Material EDA"] = results["Material"].apply(lambda x: remove_nonunique(apply_conversion(x, conversion_lower, check_lower=True)))
images = results["File"].apply(lambda x: len(find_images(f"{root_input}/{x}/Metalografie")))
results["Images"] = images

In [ ]:
idx_tah = results["Test"].apply(lambda x: np.sum([t.value == "tah" for t in x])) > 0
idx_eda = results["Material EDA"].apply(len) > 0
idx_composition = results["Composition"].apply(len) > 0
idx_images = results["Images"] > 0

print(f"Tah: {np.sum(idx_tah)}")
print(f"Tah with EDA material: {np.sum(idx_tah & idx_eda)}")
print(f"Tah with composition: {np.sum(idx_tah & idx_composition)}")
print(f"Tah with EDA material or composition: {np.sum(idx_tah & (idx_eda | idx_composition))}")
print(f"Tah with images and EDA material or composition: {np.sum(idx_tah & (idx_eda | idx_composition) & idx_images)}")

In [ ]:
eda_info = []
for _, row in results.iterrows():
    for t in row["Test"]:
        path = t.source
        for x, y in replace_source:
            path = path.replace(x, y)
        eda_info.append({"Zakazka": row["File"], "Typ": t.value, "Cesta": path})
pd.DataFrame(eda_info).sort_values(["Typ", "Zakazka"]).to_csv("eda_info.csv", index=False)

In [ ]:
results = expand_rows(results, cols_expand)
results["Composition EDA"] = results["Material EDA"].apply(lambda x: apply_conversion(x, composition))
results = results.loc[:, cols_order]

results["idx"] = results.groupby("File").cumcount()
tests["idx"] = tests.groupby("File").cumcount()

merged = pd.merge(results, tests, on=["File", "idx"], how="outer").drop(columns="idx").reset_index(drop=True)

In [ ]:
cols1 = ["File", "Test", "Material", "Material EDA", "Images", "Composition", "Composition EDA", "Specimen"]
cols2 = ["Temp.", "D0", "a0", "b0", "Du", "bU", "S0", "SU", "L0", "LU", "E", "Rp0,2", "Rt0,5", "ReH", "ReL", "Rm", "Ag", "A5", "Z", "Fm"]

merged_print = merged[cols1 + cols2]
export_dataframe(merged_print, "results.xlsx", replace_source=replace_source)

In [ ]:
import json

with open("../log.jsonl", encoding="utf-8") as f:
    entries = [json.loads(line) for line in f]

entries = pd.DataFrame(entries)
entries["type"].value_counts()

In [ ]:
import os
import pickle

import pandas as pd

files = sorted(os.path.join(root, file) for file in os.listdir(root) if file.endswith(".pkl") and "ZAK" in file)

rows = []

for file in files:
    data = pickle.load(open(file, "rb"))

    result = {attribute: [x.value for x in values] for attribute, values in data.info.items()}
    rows.append({"Zakázka": data.order_name, **result})

export = pd.DataFrame(rows)

print(f"Nalezeno souborů: {len(files)}")
print(f"Počet zakázek: {export['Zakázka'].nunique()}")
print(f"Počet řádků: {len(export)}")
print(f"Počet sloupců: {len(export.columns)}")

print("\nChybějící hodnoty:")
print(export.isna().sum())

export